In [69]:
import numpy as np
from PIL import Image
from colmap_parsing_utils import read_images_text, write_images_text, Image
from numpy.typing import NDArray
from typing import List, Literal, Optional, Tuple
import math

In [70]:
colmap_images_txt="/home/percy/data/Tanks_key5/Horse/sparse/0/images.txt"
colmap_images_full_txt="/home/percy/data/Tanks_key5/Horse/sparse/0/images_full.txt"

In [71]:
images_info=read_images_text(colmap_images_txt) 
'''
images_info is a dictionart , image_id in colmap : the attribute of images that is namedtuple
("id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids")
'''

'\nimages_info is a dictionart , image_id in colmap : the attribute of images that is namedtuple\n("id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids")\n'

In [72]:

_EPS = np.finfo(float).eps * 4.0
def unit_vector(data: NDArray, axis: Optional[int] = None) -> np.ndarray:
    """Return ndarray normalized by length, i.e. Euclidean norm, along axis.

    Args:
        axis: the axis along which to normalize into unit vector
        out: where to write out the data to. If None, returns a new np ndarray
    """
    data = np.array(data, dtype=np.float64, copy=True)
    if data.ndim == 1:
        data /= math.sqrt(np.dot(data, data))
        return data
    length = np.atleast_1d(np.sum(data * data, axis))
    np.sqrt(length, length)
    if axis is not None:
        length = np.expand_dims(length, axis)
    data /= length
    return data

def quaternion_slerp(
    quat0: NDArray, quat1: NDArray, fraction: float, spin: int = 0, shortestpath: bool = True
) -> np.ndarray:
    """Return spherical linear interpolation between two quaternions.
    Args:
        quat0: first quaternion
        quat1: second quaternion
        fraction: how much to interpolate between quat0 vs quat1 (if 0, closer to quat0; if 1, closer to quat1)
        spin: how much of an additional spin to place on the interpolation
        shortestpath: whether to return the short or long path to rotation
    """
    q0 = unit_vector(quat0[:4])
    q1 = unit_vector(quat1[:4])
    if q0 is None or q1 is None:
        raise ValueError("Input quaternions invalid.")
    if fraction == 0.0:
        return q0
    if fraction == 1.0:
        return q1
    d = np.dot(q0, q1)
    if abs(abs(d) - 1.0) < _EPS:
        return q0
    if shortestpath and d < 0.0:
        # invert rotation
        d = -d
        np.negative(q1, q1)
    angle = math.acos(d) + spin * math.pi
    if abs(angle) < _EPS:
        return q0
    isin = 1.0 / math.sin(angle)
    q0 *= math.sin((1.0 - fraction) * angle) * isin
    q1 *= math.sin(fraction * angle) * isin
    q0 += q1
    return q0

In [73]:
for colmap_id, meta_data in images_info.items():
    print(colmap_id, meta_data.xys.shape, meta_data.point3D_ids.shape)
    print(meta_data.name)

13 (3620, 2) (3620,)
000351.jpg
12 (3644, 2) (3644,)
000346.jpg
11 (3778, 2) (3778,)
000341.jpg
10 (3709, 2) (3709,)
000336.jpg
9 (3481, 2) (3481,)
000331.jpg
8 (3703, 2) (3703,)
000326.jpg
7 (3607, 2) (3607,)
000321.jpg
6 (3390, 2) (3390,)
000316.jpg
5 (3603, 2) (3603,)
000311.jpg
4 (3501, 2) (3501,)
000306.jpg
3 (3438, 2) (3438,)
000301.jpg
2 (3613, 2) (3613,)
000296.jpg
25 (3156, 2) (3156,)
000410.jpg
24 (3347, 2) (3347,)
000406.jpg
23 (3431, 2) (3431,)
000401.jpg
22 (3648, 2) (3648,)
000396.jpg
21 (3712, 2) (3712,)
000391.jpg
20 (3734, 2) (3734,)
000386.jpg
19 (3643, 2) (3643,)
000381.jpg
18 (3704, 2) (3704,)
000376.jpg
17 (3725, 2) (3725,)
000371.jpg
16 (3993, 2) (3993,)
000366.jpg
15 (3770, 2) (3770,)
000361.jpg
14 (3752, 2) (3752,)
000356.jpg
1 (3400, 2) (3400,)
000291.jpg


In [74]:
# sort the images by name
images_info = dict(sorted(images_info.items(), key=lambda item: item[1].name))
for colmap_id, meta_data in images_info.items():
    print(colmap_id, meta_data.xys.shape, meta_data.point3D_ids.shape)
    print(meta_data.name)


1 (3400, 2) (3400,)
000291.jpg
2 (3613, 2) (3613,)
000296.jpg
3 (3438, 2) (3438,)
000301.jpg
4 (3501, 2) (3501,)
000306.jpg
5 (3603, 2) (3603,)
000311.jpg
6 (3390, 2) (3390,)
000316.jpg
7 (3607, 2) (3607,)
000321.jpg
8 (3703, 2) (3703,)
000326.jpg
9 (3481, 2) (3481,)
000331.jpg
10 (3709, 2) (3709,)
000336.jpg
11 (3778, 2) (3778,)
000341.jpg
12 (3644, 2) (3644,)
000346.jpg
13 (3620, 2) (3620,)
000351.jpg
14 (3752, 2) (3752,)
000356.jpg
15 (3770, 2) (3770,)
000361.jpg
16 (3993, 2) (3993,)
000366.jpg
17 (3725, 2) (3725,)
000371.jpg
18 (3704, 2) (3704,)
000376.jpg
19 (3643, 2) (3643,)
000381.jpg
20 (3734, 2) (3734,)
000386.jpg
21 (3712, 2) (3712,)
000391.jpg
22 (3648, 2) (3648,)
000396.jpg
23 (3431, 2) (3431,)
000401.jpg
24 (3347, 2) (3347,)
000406.jpg
25 (3156, 2) (3156,)
000410.jpg


In [75]:
start_filename = images_info[1].name
end_filename = images_info[len(images_info)].name

# Extracting numbers from filenames
start_num = int(start_filename.split('.')[0])  # Convert '000291' to 291
end_num = int(end_filename.split('.')[0])     # Convert '000296' to 296

# Generate filenames in the range from start_num + 1 to end_num - 1
filenames = [f"{num:06}.jpg" for num in range(start_num, end_num+1)]

In [76]:
quaternions_before = np.array([images_info[colmap_id].qvec for colmap_id in images_info.keys()])
translation_before = np.array([images_info[colmap_id].tvec for colmap_id in images_info.keys()])
quaternions_after = []
translation_after = []
ts = np.linspace(0, 1, 6)

for i in range(0, len(quaternions_before)-2):
    quaternions_after += [quaternion_slerp(quaternions_before[i], quaternions_before[i+1], t) for t in ts][:-1]
    translation_after += [(1 - t) * translation_before[i] + t * translation_before[i+1] for t in ts][:-1]

# handcraft the last interpolation
ts = np.linspace(0, 1, 5)
quaternions_after += [quaternion_slerp(quaternions_before[i], quaternions_before[i+1], t) for t in ts]
translation_after += [(1 - t) * translation_before[i] + t * translation_before[i+1] for t in ts]


In [77]:
# len(quaternions_before), len(quaternions_after)
len(translation_before), len(translation_after)

(25, 120)

In [102]:
full_image_info = {}
# index starts from 1, there is no image_id = 0
for i in range(1, len(quaternions_after)+1):
    image_id = int(i)
    qvec = quaternions_after[i-1]
    tvec = translation_after[i-1]
    camera_id = 1   # handcraft
    image_name = filenames[i-1]
    xys = np.zeros((1, 2))  # handcraft
    point3D_ids = np.array([-1]) # handcraft
    full_image_info[i] = Image(
        id=image_id,
        qvec=qvec,
        tvec=tvec,
        camera_id=camera_id,
        name=image_name,
        xys=xys,
        point3D_ids=point3D_ids,
    )

In [103]:
full_image_info

{1: Image(id=1, qvec=array([ 0.99731047,  0.00914763, -0.07182055,  0.01139961]), tvec=array([5.55342473, 0.15883475, 1.67670798]), camera_id=1, name='000291.jpg', xys=array([[0., 0.]]), point3D_ids=array([-1])),
 2: Image(id=2, qvec=array([ 0.9974212 ,  0.00878229, -0.07034374,  0.01120648]), tvec=array([5.47684406, 0.15863419, 1.65486072]), camera_id=1, name='000292.jpg', xys=array([[0., 0.]]), point3D_ids=array([-1])),
 3: Image(id=3, qvec=array([ 0.99752956,  0.00841694, -0.06886676,  0.01101332]), tvec=array([5.40026339, 0.15843362, 1.63301345]), camera_id=1, name='000293.jpg', xys=array([[0., 0.]]), point3D_ids=array([-1])),
 4: Image(id=4, qvec=array([ 0.99763557,  0.00805157, -0.06738963,  0.01082014]), tvec=array([5.32368272, 0.15823305, 1.61116619]), camera_id=1, name='000294.jpg', xys=array([[0., 0.]]), point3D_ids=array([-1])),
 5: Image(id=5, qvec=array([ 0.99773922,  0.00768618, -0.06591233,  0.01062693]), tvec=array([5.24710205, 0.15803249, 1.58931893]), camera_id=1, nam

In [104]:
write_images_text(full_image_info, colmap_images_full_txt)